<a href="https://colab.research.google.com/github/balloontip/deep-learning/blob/main/chapter-08/08-08-Project-Transformer-English-to-Turkish-Neural-Machine-Translation-Project.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Goal: Build, train, and evaluate a Transformer model from scratch in PyTorch to translate English sentences into Turkish. This end-to-end neural machine translation (NMT) project demonstrates text tokenization, vocabulary construction, sequence padding, positional encoding, multi-head self-attention, encoder and decoder layers, teacher forcing, sequence-to-sequence training with Cross-Entropy Loss and the Adam optimizer, gradient clipping, greedy decoding, beam search decoding, translation of both training and unseen sentences, vocabulary analysis, and evaluation of the model's translation capabilities and limitations.

## 1. Import Required Libraries

Let's import libraries for building, training, and running our Transformer.
You'll see PyTorch, math, and random—all essential for deep learning and working with text data.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random

## 2. Set Hyperparameters and Special Tokens

We define:
- **Model parameters** (dimensions, layers, heads, etc.)
- **Training parameters** (batch size, learning rate, epochs)
- **Special tokens** ([PAD], [SOS], [EOS], [UNK]) for marking padding, start, end, and unknown words.

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model hyperparameters
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 3
D_FF = 2048
DROPOUT = 0.1
MAX_SEQ_LEN = 100

# Training
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20

# Tokens
PAD_TOKEN = '[PAD]'
SOS_TOKEN = '[SOS]'
EOS_TOKEN = '[EOS]'
UNK_TOKEN = '[UNK]'

## 3. Prepare English-Turkish Sample Dataset

For demonstration, we'll use a small set of English–Turkish phrase pairs.
(A real translation model needs thousands or millions of sentences. Here, we keep it small and simple for learning.)

In [ ]:
raw_data = [
    ("Hello", "Merhaba"),
    ("How are you?", "Nasılsın?"),
    ("I love tomatoes", "Domatesleri severim"),
    ("The dog chased the mouse", "Köpek fareyi kovaladı"),
    ("Thank you", "Teşekkür ederim"),
    ("Good morning", "Günaydın"),
    ("See you later", "Görüşürüz"),
    ("What is your name?", "Adın ne?"),
    ("My name is John", "Adım John"),
    ("I am a student", "Ben bir öğrenciyim"),
    ("This is a book", "Bu bir kitap"),
    ("She is happy", "O mutlu"),
    ("We are learning", "Öğreniyoruz"),
    ("He runs fast", "Hızlı koşar"),
    ("They play football", "Futbol oynuyorlar"),
    ("Where is the station?", "İstasyon nerede?"),
    ("I need help", "Yardıma ihtiyacım var"),
    ("Please speak slowly", "Lütfen yavaş konuşun"),
    ("I understand", "Anlıyorum"),
    ("I don't understand", "Anlamıyorum"),
    ("Can you repeat that?", "Tekrar edebilir misiniz?"),
    ("How much is this?", "Bu ne kadar?"),
    ("I am hungry", "Açım"),
    ("I am thirsty", "Susamışım"),
    ("It is cold today", "Bugün hava soğuk"),
    ("It is hot today", "Bugün hava sıcak"),
    ("I want to eat pizza", "Pizza yemek istiyorum"),
    ("Do you speak English?", "İngilizce konuşuyor musunuz?"),
    ("Yes, a little", "Evet, biraz"),
    ("No, not much", "Hayır, pek değil")
]

## 4. Tokenization, Vocabulary Building, and Padding

Transform plain sentences into token lists, associate tokens to indices, and pad sequences to the same length.
We also add [SOS] (start of sentence) and [EOS] (end) tokens for the target side.

In [ ]:
def tokenize(text):
    return text.lower().split()

def build_vocabulary(tokenized_sentences):
    word_to_idx = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
    idx_to_word = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN, 3: UNK_TOKEN}
    current_idx = 4
    for sentence in tokenized_sentences:
        for word in sentence:
            if word not in word_to_idx:
                word_to_idx[word] = current_idx
                idx_to_word[current_idx] = word
                current_idx += 1
    return word_to_idx, idx_to_word

def prepare_data(data, max_seq_len):
    src_sentences = [item[0] for item in data]
    tgt_sentences = [item[1] for item in data]

    tokenized_src = [tokenize(s) for s in src_sentences]
    tokenized_tgt = [tokenize(s) for s in tgt_sentences]

    src_word_to_idx, src_idx_to_word = build_vocabulary(tokenized_src)
    tgt_word_to_idx, tgt_idx_to_word = build_vocabulary(tokenized_tgt)

    # Encode tokens and add SOS/EOS where needed
    src_data, tgt_input_data, tgt_output_data = [], [], []
    for i in range(len(tokenized_src)):
        src_seq = [src_word_to_idx.get(word, src_word_to_idx[UNK_TOKEN]) for word in tokenized_src[i]]
        tgt_seq = [tgt_word_to_idx.get(word, tgt_word_to_idx[UNK_TOKEN]) for word in tokenized_tgt[i]]

        tgt_input_seq = [tgt_word_to_idx[SOS_TOKEN]] + tgt_seq
        tgt_output_seq = tgt_seq + [tgt_word_to_idx[EOS_TOKEN]]

        src_data.append(src_seq)
        tgt_input_data.append(tgt_input_seq)
        tgt_output_data.append(tgt_output_seq)

    # Pad sequences
    padded_src = torch.zeros((len(src_data), max_seq_len), dtype=torch.long)
    padded_tgt_input = torch.zeros((len(tgt_input_data), max_seq_len), dtype=torch.long)
    padded_tgt_output = torch.zeros((len(tgt_output_data), max_seq_len), dtype=torch.long)

    for i in range(len(src_data)):
        length_src = min(len(src_data[i]), max_seq_len)
        length_tgt_in = min(len(tgt_input_data[i]), max_seq_len)
        length_tgt_out = min(len(tgt_output_data[i]), max_seq_len)

        padded_src[i, :length_src] = torch.tensor(src_data[i][:length_src])
        padded_tgt_input[i, :length_tgt_in] = torch.tensor(tgt_input_data[i][:length_tgt_in])
        padded_tgt_output[i, :length_tgt_out] = torch.tensor(tgt_output_data[i][:length_tgt_out])

    return padded_src, padded_tgt_input, padded_tgt_output, src_word_to_idx, src_idx_to_word, tgt_word_to_idx, tgt_idx_to_word

# Prepare the data
src_padded, tgt_input_padded, tgt_output_padded, src_word_to_idx, src_idx_to_word, tgt_word_to_idx, tgt_idx_to_word = prepare_data(raw_data, MAX_SEQ_LEN)

## 5. Define Transformer Model Parts

This is a modular breakdown of the Transformer:
- **PositionalEncoding**: Adds information about token order
- **MultiHeadAttention**: Finds relationships between all words in a sequence
- **FeedForward**: Processes information nonlinearly
- **Encoder and Decoder Layers**: Stack attention and feedforward for English (source) and Turkish (target) sequences
- **Full Transformer**: Stacks encoder and decoder for translation

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.d_model = d_model
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(1)
        query = self.wq(query).view(-1, batch_size, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        key = self.wk(key).view(-1, batch_size, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        value = self.wv(value).view(-1, batch_size, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.permute(1, 0, 2, 3)
            scores = scores.masked_fill(mask == 0, -1e9)
            scores = scores.permute(1, 0, 2, 3)
        weights = torch.softmax(scores, dim=-1)
        attention_output = torch.matmul(weights, value)
        attention_output = attention_output.permute(2, 1, 0, 3).contiguous().view(-1, batch_size, self.d_model)
        output = self.wo(attention_output)
        return output

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.w_2(self.dropout(self.relu(self.w_1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_mask):
        attn_output = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout1(attn_output))
        ff_output = self.feed_forward(src)
        src = self.norm2(src + self.dropout2(ff_output))
        return src

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask):
        src = self.embedding(src)
        src = self.dropout(self.pos_encoder(src))
        for layer in self.layers:
            src = layer(src, src_mask)
        return src

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.encoder_decoder_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, encoder_output, tgt_mask, src_tgt_mask):
        attn_output = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(attn_output))
        attn_output = self.encoder_decoder_attn(tgt, encoder_output, encoder_output, src_tgt_mask)
        tgt = self.norm2(tgt + self.dropout2(attn_output))
        ff_output = self.feed_forward(tgt)
        tgt = self.norm3(tgt + self.dropout3(ff_output))
        return tgt

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.linear_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, encoder_output, tgt_mask, src_tgt_mask):
        tgt = self.embedding(tgt)
        tgt = self.dropout(self.pos_encoder(tgt))
        for layer in self.layers:
            tgt = layer(tgt, encoder_output, tgt_mask, src_tgt_mask)
        output = self.linear_out(tgt)
        return output

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout):
        super(Transformer, self).__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, dropout)

    def make_src_mask(self, src):
        src_mask = (src != src_word_to_idx[PAD_TOKEN]).unsqueeze(1).unsqueeze(2)
        return src_mask

    def make_tgt_mask(self, tgt):
        tgt_pad_mask = (tgt != tgt_word_to_idx[PAD_TOKEN]).unsqueeze(1).unsqueeze(2)
        tgt_seq_len = tgt.size(1)
        tgt_lookahead_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len), diagonal=1).bool().to(device)
        tgt_lookahead_mask = tgt_lookahead_mask.unsqueeze(0).unsqueeze(0)
        tgt_mask = tgt_pad_mask & (~tgt_lookahead_mask)
        return tgt_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        src_tgt_mask = src_mask
        src = src.transpose(0, 1)
        tgt = tgt.transpose(0, 1)
        encoder_output = self.encoder(src, src_mask)
        decoder_output = self.decoder(tgt, encoder_output, tgt_mask, src_tgt_mask)
        return decoder_output.transpose(0, 1)

## 6. Batch Preparation for Training

For each epoch, we randomly shuffle and batch the data to improve the robustness of training.

In [ ]:
def create_batches(src_data, tgt_input_data, tgt_output_data, batch_size):
    dataset_size = src_data.size(0)
    indices = list(range(dataset_size))
    random.shuffle(indices)
    batches = []
    for i in range(0, dataset_size, batch_size):
        batch_indices = indices[i:i + batch_size]
        src_batch = src_data[batch_indices].to(device)
        tgt_input_batch = tgt_input_data[batch_indices].to(device)
        tgt_output_batch = tgt_output_data[batch_indices].to(device)
        batches.append((src_batch, tgt_input_batch, tgt_output_batch))
    return batches

## 7. Train the Transformer for Translation
The core learning loop!  
1. Create the model  
2. Specify loss function (ignore padding tokens)  
3. Train for set epochs on shuffled batches, using Adam optimizer  
4. Print the loss to see improvement

In [ ]:
src_vocab_size = len(src_word_to_idx)
tgt_vocab_size = len(tgt_word_to_idx)

model = Transformer(src_vocab_size, tgt_vocab_size, D_MODEL, NUM_LAYERS, NUM_HEADS, D_FF, DROPOUT).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_word_to_idx[PAD_TOKEN])
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    batches = create_batches(src_padded, tgt_input_padded, tgt_output_padded, BATCH_SIZE)
    for src_batch, tgt_input_batch, tgt_output_batch in batches:
        output = model(src_batch, tgt_input_batch)
        loss = criterion(output.reshape(-1, tgt_vocab_size), tgt_output_batch.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(batches)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Average Loss: {avg_loss:.4f}")

print("Training complete!")

Starting training...
Epoch 1/20, Average Loss: 4.5131
Epoch 2/20, Average Loss: 3.5631
Epoch 3/20, Average Loss: 3.2773
Epoch 4/20, Average Loss: 2.9084
Epoch 5/20, Average Loss: 2.6656
Epoch 6/20, Average Loss: 2.4513
Epoch 7/20, Average Loss: 2.1679
Epoch 8/20, Average Loss: 1.9064
Epoch 9/20, Average Loss: 1.6124
Epoch 10/20, Average Loss: 1.3890
Epoch 11/20, Average Loss: 1.1835
Epoch 12/20, Average Loss: 0.9863
Epoch 13/20, Average Loss: 0.8391
Epoch 14/20, Average Loss: 0.6605
Epoch 15/20, Average Loss: 0.5772
Epoch 16/20, Average Loss: 0.4758
Epoch 17/20, Average Loss: 0.3876
Epoch 18/20, Average Loss: 0.2967
Epoch 19/20, Average Loss: 0.2507
Epoch 20/20, Average Loss: 0.1905
Training complete!


## 8. Simple Translation Function (Greedy Decoding)

Now let's create a function to translate new English sentences to Turkish!

**Greedy Decoding**: At each step, we pick the word with the highest probability. This is simple but not always the best approach for translation quality.

In [ ]:
def translate_sentence(model, src_sentence, src_word_to_idx, tgt_word_to_idx, tgt_idx_to_word, max_len=MAX_SEQ_LEN):
    model.eval()

    # Tokenize and encode source sentence
    src_tokens = tokenize(src_sentence)
    src_indices = [src_word_to_idx.get(word, src_word_to_idx[UNK_TOKEN]) for word in src_tokens]

    # Pad source sequence
    src_tensor = torch.zeros(1, max_len, dtype=torch.long).to(device)
    src_len = min(len(src_indices), max_len)
    src_tensor[0, :src_len] = torch.tensor(src_indices[:src_len])

    # Start with SOS token for target
    tgt_indices = [tgt_word_to_idx[SOS_TOKEN]]

    # Generate translation word by word
    with torch.no_grad():
        for _ in range(max_len - 1):
            tgt_tensor = torch.zeros(1, max_len, dtype=torch.long).to(device)
            tgt_len = min(len(tgt_indices), max_len)
            tgt_tensor[0, :tgt_len] = torch.tensor(tgt_indices[:tgt_len])

            # Get model predictions
            output = model(src_tensor, tgt_tensor)

            # Get next word prediction (highest probability)
            next_word_idx = output[0, tgt_len-1].argmax().item()

            # Stop if we generate EOS token
            if next_word_idx == tgt_word_to_idx[EOS_TOKEN]:
                break

            tgt_indices.append(next_word_idx)

    # Convert indices back to words
    translated_words = []
    for idx in tgt_indices[1:]:  # Skip SOS token
        if idx == tgt_word_to_idx[EOS_TOKEN]:
            break
        translated_words.append(tgt_idx_to_word.get(idx, UNK_TOKEN))

    return ' '.join(translated_words)

print("Translation function ready!")

Translation function ready!


## 9. Beam Search for Better Translation Quality

**Beam Search** is smarter than greedy decoding. Instead of always picking the single best word, it keeps track of multiple possible translations and picks the overall best sequence.

**Key Concept**: We maintain a "beam" of the top-k most promising partial translations at each step.

In [ ]:
def beam_search_translate(model, src_sentence, src_word_to_idx, tgt_word_to_idx, tgt_idx_to_word, beam_size=3, max_len=MAX_SEQ_LEN):
    model.eval()

    # Prepare source sentence
    src_tokens = tokenize(src_sentence)
    src_indices = [src_word_to_idx.get(word, src_word_to_idx[UNK_TOKEN]) for word in src_tokens]
    src_tensor = torch.zeros(1, max_len, dtype=torch.long).to(device)
    src_len = min(len(src_indices), max_len)
    src_tensor[0, :src_len] = torch.tensor(src_indices[:src_len])

    # Initialize beam with SOS token
    # Each beam item: (sequence, log_probability)
    beams = [([tgt_word_to_idx[SOS_TOKEN]], 0.0)]
    completed_beams = []

    with torch.no_grad():
        for step in range(max_len - 1):
            candidates = []

            for sequence, log_prob in beams:
                if sequence[-1] == tgt_word_to_idx[EOS_TOKEN]:
                    completed_beams.append((sequence, log_prob))
                    continue

                # Prepare target tensor for current sequence
                tgt_tensor = torch.zeros(1, max_len, dtype=torch.long).to(device)
                seq_len = min(len(sequence), max_len)
                tgt_tensor[0, :seq_len] = torch.tensor(sequence[:seq_len])

                # Get model predictions
                output = model(src_tensor, tgt_tensor)
                log_probs = torch.log_softmax(output[0, seq_len-1], dim=0)

                # Get top beam_size candidates
                top_log_probs, top_indices = log_probs.topk(beam_size)

                for i in range(beam_size):
                    new_sequence = sequence + [top_indices[i].item()]
                    new_log_prob = log_prob + top_log_probs[i].item()
                    candidates.append((new_sequence, new_log_prob))

            # Keep top beam_size candidates
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:beam_size]

            # Stop if all beams are completed
            if not beams:
                break

    # Add remaining beams to completed
    completed_beams.extend(beams)

    # Select best completed beam
    if completed_beams:
        best_sequence, _ = max(completed_beams, key=lambda x: x[1])
        translated_words = []
        for idx in best_sequence[1:]:  # Skip SOS token
            if idx == tgt_word_to_idx[EOS_TOKEN]:
                break
            translated_words.append(tgt_idx_to_word.get(idx, UNK_TOKEN))
        return ' '.join(translated_words)

    return "Translation failed"

print("Beam search translation ready!")

Beam search translation ready!


## 10. Test Model on Training Examples

Let's see how well our model learned to translate the training sentences.
Since we used a small dataset, the model might memorize these examples pretty well!

In [ ]:
print("--- Testing on Training Data ---\n")

# Test on first 8 training examples
test_indices = [0, 1, 2, 3, 4, 5, 6, 7]

print("English → Turkish Translations:")
print("=" * 60)

for i in test_indices:
    english_sentence = raw_data[i][0]
    actual_turkish = raw_data[i][1]

    # Try both translation methods
    greedy_translation = translate_sentence(model, english_sentence, src_word_to_idx, tgt_word_to_idx, tgt_idx_to_word)
    beam_translation = beam_search_translate(model, english_sentence, src_word_to_idx, tgt_word_to_idx, tgt_idx_to_word)

    print(f"English:     {english_sentence}")
    print(f"Actual:      {actual_turkish}")
    print(f"Greedy:      {greedy_translation}")
    print(f"Beam Search: {beam_translation}")
    print("-" * 60)

--- Testing on Training Data ---

English → Turkish Translations:
English:     Hello
Actual:      Merhaba
Greedy:      merhaba
Beam Search: merhaba
------------------------------------------------------------
English:     How are you?
Actual:      Nasılsın?
Greedy:      nasılsın?
Beam Search: nasılsın?
------------------------------------------------------------
English:     I love tomatoes
Actual:      Domatesleri severim
Greedy:      domatesleri severim
Beam Search: domatesleri severim
------------------------------------------------------------
English:     The dog chased the mouse
Actual:      Köpek fareyi kovaladı
Greedy:      köpek fareyi kovaladı
Beam Search: köpek fareyi kovaladı
------------------------------------------------------------
English:     Thank you
Actual:      Teşekkür ederim
Greedy:      teşekkür ederim
Beam Search: teşekkür ederim
------------------------------------------------------------
English:     Good morning
Actual:      Günaydın
Greedy:      günaydın
B

## 11. Test Translation on New Sentences

The real challenge: Can our model translate sentences it has never seen before?

**Note**: Since we trained on a very small dataset, the model's vocabulary is limited. It can only translate words it has learned, and may struggle with completely new sentences.

In [ ]:
print("--- Testing on New Sentences ---\n")

# Test sentences that weren't in our training data
new_test_sentences = [
    "I am happy",           # Uses known words in new combination
    "Thank you very much",  # Partially known words
    "Good night",           # Completely new phrase
    "How are you today?",   # Mix of known/unknown
    "I love you"            # Simple but not in training
]

print("New Sentence Translation Tests:")
print("=" * 50)

for sentence in new_test_sentences:
    print(f"English: {sentence}")

    try:
        greedy_result = translate_sentence(model, sentence, src_word_to_idx, tgt_word_to_idx, tgt_idx_to_word)
        beam_result = beam_search_translate(model, sentence, src_word_to_idx, tgt_word_to_idx, tgt_idx_to_word)

        print(f"Greedy:      {greedy_result}")
        print(f"Beam Search: {beam_result}")
    except Exception as e:
        print(f"Translation error: {e}")

    print("-" * 50)

# Show unknown words in test sentences
print("\n--- Vocabulary Analysis ---")
print("Words in test sentences that weren't in training data:")

all_test_words = set()
for sentence in new_test_sentences:
    words = tokenize(sentence)
    all_test_words.update(words)

unknown_words = []
for word in all_test_words:
    if word not in src_word_to_idx:
        unknown_words.append(word)

if unknown_words:
    print(f"Unknown words: {unknown_words}")
    print("These will be replaced with [UNK] tokens during translation.")
else:
    print("All words are known from training data!")

--- Testing on New Sentences ---

New Sentence Translation Tests:
English: I am happy
Greedy:      açım
Beam Search: açım
--------------------------------------------------
English: Thank you very much
Greedy:      teşekkür ederim
Beam Search: teşekkür ederim
--------------------------------------------------
English: Good night
Greedy:      günaydın
Beam Search: günaydın
--------------------------------------------------
English: How are you today?
Greedy:      nasılsın?
Beam Search: nasılsın?
--------------------------------------------------
English: I love you
Greedy:      domatesleri severim
Beam Search: domatesleri severim
--------------------------------------------------

--- Vocabulary Analysis ---
Words in test sentences that weren't in training data:
Unknown words: ['night', 'today?', 'very']
These will be replaced with [UNK] tokens during translation.


## 12. Analyze What the Model Learned

Let's examine our model's vocabulary and understand its limitations and capabilities.

This helps us understand why certain translations work well while others don't.

In [ ]:
print("--- Model Vocabulary Analysis ---\n")

print(f"English Vocabulary Size: {len(src_word_to_idx)}")
print(f"Turkish Vocabulary Size: {len(tgt_word_to_idx)}")

print(f"\nEnglish vocabulary (first 20 words):")
english_words = [word for word in src_word_to_idx.keys() if word not in [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]]
for i, word in enumerate(english_words[:20]):
    print(f"{i+1:2d}. {word}")

print(f"\nTurkish vocabulary (first 20 words):")
turkish_words = [word for word in tgt_word_to_idx.keys() if word not in [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]]
for i, word in enumerate(turkish_words[:20]):
    print(f"{i+1:2d}. {word}")

# Show model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Statistics:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")

# Show training data statistics
print(f"\nTraining Data Statistics:")
print(f"Number of sentence pairs: {len(raw_data)}")
print(f"Maximum sequence length: {MAX_SEQ_LEN}")

avg_src_len = sum(len(tokenize(pair[0])) for pair in raw_data) / len(raw_data)
avg_tgt_len = sum(len(tokenize(pair[1])) for pair in raw_data) / len(raw_data)

print(f"Average English sentence length: {avg_src_len:.2f} words")
print(f"Average Turkish sentence length: {avg_tgt_len:.2f} words")

--- Model Vocabulary Analysis ---

English Vocabulary Size: 73
Turkish Vocabulary Size: 63

English vocabulary (first 20 words):
 1. hello
 2. how
 3. are
 4. you?
 5. i
 6. love
 7. tomatoes
 8. the
 9. dog
10. chased
11. mouse
12. thank
13. you
14. good
15. morning
16. see
17. later
18. what
19. is
20. your

Turkish vocabulary (first 20 words):
 1. merhaba
 2. nasılsın?
 3. domatesleri
 4. severim
 5. köpek
 6. fareyi
 7. kovaladı
 8. teşekkür
 9. ederim
10. günaydın
11. görüşürüz
12. adın
13. ne?
14. adım
15. john
16. ben
17. bir
18. öğrenciyim
19. bu
20. kitap

Model Statistics:
Total parameters: 22,171,199
Trainable parameters: 22,171,199
Model size: ~84.58 MB (float32)

Training Data Statistics:
Number of sentence pairs: 30
Maximum sequence length: 100
Average English sentence length: 3.30 words
Average Turkish sentence length: 2.10 words


## 13. Understanding Results and Next Steps

**What you've accomplished:**
- Built a complete Transformer model from scratch
- Learned about attention mechanisms and positional encoding
- Implemented both greedy and beam search decoding
- Created an English-Turkish translation system

**Why some translations might not be perfect:**
- **Small dataset**: Only 30 sentence pairs (real models use millions!)
- **Limited vocabulary**: Model can only translate words it has seen
- **Simple training**: Real models train for days/weeks on powerful GPUs

**To improve translation quality:**
- Use larger, more diverse datasets
- Train for more epochs with learning rate scheduling  
- Add techniques like label smoothing and warmup
- Use pre-trained embeddings (like Word2Vec)
- Implement more sophisticated beam search variants


## **Summary**

**What you've learned about Transformer models:**

**Architecture Understanding**: Built the complete Transformer from scratch, understanding how encoder-decoder attention enables translation between languages.

**Key Components Mastered**: Positional encoding for sequence order, multi-head attention for finding word relationships, and feed-forward networks for processing information.

**Text Processing Pipeline**: From raw sentences to tokens to embeddings, including handling special tokens and padding for batched training.

**Training Process**: How to train sequence-to-sequence models with teacher forcing and handle variable-length sequences efficiently.

**Decoding Strategies**: Both greedy decoding (simple but fast) and beam search (better quality but slower) for generating translations.

**Real-world Applications**: This foundation prepares you for working with modern NLP models like BERT, GPT, and T5, as well as building chatbots, summarization systems, and other language AI applications.

The Transformer architecture you've just implemented is the foundation behind ChatGPT, Google Translate, and most modern language AI systems!